#### Memory

prove the model has no memory, then build some by hand, then let a checkpointer do it. Measure what remembering costs, trim or summarise to afford it, and split short-term memory from long-term.

In [1]:
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()
model  = ChatOpenRouter(model = "nvidia/nemotron-3.5-lightning:free", temperature = 0)

#### 1. Prove the model remembers nothing
Two calls, one after the other. The second one asks about the first.

In [2]:
model.invoke("My name is Sayali.").text

'Hello Sayali! Nice to meet you. How can I help you today?'

In [3]:
model.invoke("What is my name?").text

"I don't know your name! I don't have access to personal information about you unless you share it. If you'd like, you can tell me your name, and I'll remember it for the rest of our conversation. 😊"

It doesn't know. Nothing was stored, each invoke is a separate request to a server that keeps nothing. The model is a function: text in, text out.

Everything today is about you doing the remembering, then handing that job off.

#### 2. Keep the conversation yourself, in a list
Put the conversation in the request: a list, appended to.

In [4]:
from langchain.messages import HumanMessage

history = []

def say(text):
    history.append(HumanMessage(text))
    reply = model.invoke(history)
    history.append(reply)
    return reply.text

In [5]:
say("My name is Sayali and I use a wheelchair.")

"Nice to meet you, Sayali! It's great to connect. Using a wheelchair is just one part of your story—whatever you'd like to share, discuss, or ask about, I'm here to help. How can I support you today?"

In [6]:
history

[HumanMessage(content='My name is Sayali and I use a wheelchair.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Nice to meet you, Sayali! It's great to connect. Using a wheelchair is just one part of your story—whatever you'd like to share, discuss, or ask about, I'm here to help. How can I support you today?", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My name is Sayali and I use a wheelchair."\n   - This is a self-introduction with a disability disclosure.\n   - The user is likely looking for acknowledgment, support, conversation, or just sharing information.\n\n2.  **Identify Key Elements:**\n   - Name: Sayali\n   - Uses a wheelchair\n   - No specific question or request yet\n\n3.  **Determine Appropriate Response:**\n   - Acknowledge the introduction warmly\n   - Respect the disability context without making assumptions\n   - Offer open-ended support or ask how I can help\n   - Keep it in

In [7]:
say("Book me a table for two on Friday.")

"I'd love to help, but I can't make actual restaurant reservations—I don't have access to booking systems or the ability to call businesses directly. \n\nHowever, I can definitely point you in the right direction! When you book, whether by phone or through an app like OpenTable, Resy, or the restaurant's website, it's a great idea to ask about accessibility so the setup works well for you. Consider asking things like:\n- Is there step-free access from the entrance to the dining area?\n- Are tables spaced for wheelchair maneuverability?\n- Is the restroom accessible?\n- Can they accommodate a specific seating preference (e.g., higher tables, corner spots)?\n\nIf you tell me your city or neighborhood, I can help you find accessible restaurants nearby or draft a quick script to use when calling. Just let me know!"

In [8]:
history


[HumanMessage(content='My name is Sayali and I use a wheelchair.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Nice to meet you, Sayali! It's great to connect. Using a wheelchair is just one part of your story—whatever you'd like to share, discuss, or ask about, I'm here to help. How can I support you today?", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My name is Sayali and I use a wheelchair."\n   - This is a self-introduction with a disability disclosure.\n   - The user is likely looking for acknowledgment, support, conversation, or just sharing information.\n\n2.  **Identify Key Elements:**\n   - Name: Sayali\n   - Uses a wheelchair\n   - No specific question or request yet\n\n3.  **Determine Appropriate Response:**\n   - Acknowledge the introduction warmly\n   - Respect the disability context without making assumptions\n   - Offer open-ended support or ask how I can help\n   - Keep it in

In [9]:
len(history)

4

Four messages: two from you, two from the model. Everything below is this same list, with the bookkeeping done for you, or made cheaper.

#### 3. Hand the list to a checkpointer
create_agent with a **checkpointer** does the appending for you.

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(
    model = model,
    tools = [],
    system_prompt = "You are a helpful restaurant booking assistant.",
    checkpointer = InMemorySaver(),
)

tools=[] for now, Day 7 fills that in. A checkpointer needs to know which conversation it's saving, that's thread_id.

In [ ]:
model.invoke('you pass your message')

assistant.invoke({'messages': ['message']})

In [18]:
sayali = {"configurable": {"thread_id": "sayali"}}
assistant.invoke({"messages": [HumanMessage('my name is sayali')]}, config=sayali)

{'messages': [HumanMessage(content='my name is sayali', additional_kwargs={}, response_metadata={}, id='7cdc113c-1951-4376-903a-1f83614754dc'),
  AIMessage(content="Hello Sayali! Nice to meet you. I'm your restaurant booking assistant. How can I help you today—whether it's finding a restaurant, making a reservation, checking availability, or something else? 😊", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "my name is sayali"\n   - This is a simple personal introduction.\n   - Context: I\'m a restaurant booking assistant.\n\n2.  **Determine Appropriate Response:**\n   - Acknowledge the name.\n   - Since I\'m a restaurant booking assistant, I should tie it into the context politely.\n   - Ask how I can help with restaurant booking/reservation.\n   - Keep it friendly and professional.\n\n3.  **Draft Response (mental):**\n   "Hello Sayali! Nice to meet you. I\'m your restaurant booking assistant. How can I help you tod

In [22]:
sayali = {"configurable":{"thread_id":"sayali"}}

def chat(text, config):
    result = assistant.invoke({"messages":[HumanMessage(text)]}, config = config)
    return result["messages"][-1].text

In [23]:
chat("My name is Sayali and I use a wheelchair.", sayali)

"Nice to meet you, Sayali! I'll make sure your wheelchair accessibility is taken into account when finding a restaurant for you. \n\nTo get started with your booking, could you let me know:\n- What date and time would you like to dine?\n- How many people will be in your party?\n- Any specific preferences (cuisine, location, etc.)?\n\nI'll then look for fully accessible options that work for you."

In [24]:
chat("Book me a table for two on Friday.",sayali)

"Hey Sayali! I'd be happy to book a table for two on Friday. To make sure I find a fully accessible restaurant that works with your wheelchair, I just need a few more details:\n\n- **Which Friday** (the specific date)?\n- **What time** would you like to dine?\n- **Any preferred area or cuisine**?\n- Do you have a specific neighborhood or city in mind?\n\nOnce I have those, I'll search for wheelchair-accessible options and get your booking confirmed."

No history variable anywhere in your code. The checkpointer kept it.


#### 4. thread_id separates one conversation from another

In [27]:
pooja = {"configurable": {"thread_id": "pooja"}}

chat("What is my name?", pooja)

"I don't know your name, and I don't have access to any personal information about you. You can call me whatever you'd like, or feel free to share your name if you'd like! I'm here to help with restaurant bookings or anything else you need. 😊"

In [28]:

chat("What is my name?", sayali)

"Your name is **Sayali**, as you mentioned at the very start of our conversation! \n\nIs there anything else I can help you with for your Friday table booking? I'm ready to find a fully accessible restaurant once you let me know the date, time, and any location or cuisine preferences."

Pooja gets nothing, Sayali's thread still knows.

thread_id is the entire privacy boundary between two conversations, and it's a string you chose. Get it wrong and one user reads another's chat.

#### 5. Inspect the saved conversation with get_state

In [29]:
state = assistant.get_state(sayali)
state

StateSnapshot(values={'messages': [HumanMessage(content='My name is Sayali and I use a wheelchair.', additional_kwargs={}, response_metadata={}, id='65c4271f-4053-4fcf-88fa-91d0abbf8a5d'), AIMessage(content="Nice to meet you, Sayali! I'll make sure your wheelchair accessibility is taken into account when finding a restaurant for you. \n\nTo get started with your booking, could you let me know:\n- What date and time would you like to dine?\n- How many people will be in your party?\n- Any specific preferences (cuisine, location, etc.)?\n\nI'll then look for fully accessible options that work for you.", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My name is Sayali and I use a wheelchair."\n   - This is a personal introduction with an accessibility need.\n   - The context is a restaurant booking assistant scenario.\n\n2.  **Identify Intent/Goal:**\n   - The user is likely setting up for a restaurant booking.\n   - Th

In [30]:
for m in state.values["messages"]:
    m.pretty_print()

================================ Human Message =================================

My name is Sayali and I use a wheelchair.
================================== Ai Message ==================================

Nice to meet you, Sayali! I'll make sure your wheelchair accessibility is taken into account when finding a restaurant for you. 

To get started with your booking, could you let me know:
- What date and time would you like to dine?
- How many people will be in your party?
- Any specific preferences (cuisine, location, etc.)?

I'll then look for fully accessible options that work for you.
================================ Human Message =================================

Book me a table for two on Friday.
================================== Ai Message ==================================

Hey Sayali! I'd be happy to book a table for two on Friday. To make sure I find a fully accessible restaurant that works with your wheelchair, I just need a few more details:

- **Which Friday** (the spec

#### 6. Measure how input tokens grow every turn
Every turn sends the whole list again, not a figure of speech. The reply carries the token counts.

In [31]:
import time

usage = []

def chat_tracked(text, config):
    # time.sleep(6)                 # the free tier limits calls per minute
    result = assistant.invoke({"messages": [HumanMessage(text)]}, config = config)
    last = result["messages"][-1]
    usage.append(last.usage_metadata["input_tokens"])

The sleep isn't part of the lesson, it just keeps back-to-back turns from tripping the free tier's per-minute limit.

In [32]:
long_chat = {"configurable": {"thread_id": "cost_demo"}}

chat_tracked("I want to book a table.", long_chat)
chat_tracked("Make it Friday.", long_chat)
chat_tracked("For four people.", long_chat)
chat_tracked("Around eight in the evening.", long_chat)

usage

[31, 134, 253, 359]

Four short questions, and the input grew every turn. Imagine forty turns on a paid model: the last message costs the most and says the least.

Two ways out, and they lose different things.

#### 7. Fix one: drop old messages with trim_messages
trim_messages walks the list from the end backwards and cuts when the budget runs out. The simplest budget is a message count: pass len as the counter.

In [33]:
from langchain.messages import trim_messages

full = assistant.get_state(long_chat).values["messages"]

full


[HumanMessage(content='I want to book a table.', additional_kwargs={}, response_metadata={}, id='50fe9914-7fe7-44ea-ace5-5f3d7dbef0f7'),
 AIMessage(content="I'd be happy to help you book a table! To get everything set up, could you let me know:\n\n- **Which restaurant** you'd like to book at\n- **Date and time** you're looking for\n- **Number of guests**\n- **Any special requests** (dietary restrictions, occasion, accessibility needs, etc.)\n\nOnce I have those details, I'll check availability and confirm your reservation.", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** User says "I want to book a table."\n2.  **Identify Intent:** The user wants to make a restaurant reservation.\n3.  **Determine Missing Information:** To book a table, I need several details:\n   - Restaurant name/location\n   - Date and time\n   - Number of people\n   - Any special requirements (dietary restrictions, occasion, etc.)\n   - Contact information (someti

In [34]:
kept = trim_messages(
    full,
    max_tokens = 4,
    token_counter = len,
    strategy = "last",
    start_on="human"
)
kept

[HumanMessage(content='For four people.', additional_kwargs={}, response_metadata={}, id='305bd465-5dfb-462d-8c09-9d1cb5be94e8'),
 AIMessage(content="Thanks! Four people noted. To check availability and book your table, I just need a few more details:\n\n- **Which restaurant** are you looking at?\n- **Which Friday** (specific date)?\n- **What time** works best for you?\n- Any **special requests** (dietary needs, occasion, accessibility, etc.)?\n\nOnce I have those, I'll check availability and confirm your reservation right away!", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "For four people."\n   - Context: They want to book a table, previously mentioned "Friday", and now specifying party size: 4 people.\n   - Missing info: Restaurant name, specific date (which Friday?), time, any special requests.\n\n2.  **Identify Missing Requirements:**\n   - To actually book a table, I need: Restaurant name, Date (specific Fri

In [35]:
for m in kept:
    m.pretty_print()

================================ Human Message =================================

For four people.
================================== Ai Message ==================================

Thanks! Four people noted. To check availability and book your table, I just need a few more details:

- **Which restaurant** are you looking at?
- **Which Friday** (specific date)?
- **What time** works best for you?
- Any **special requests** (dietary needs, occasion, accessibility, etc.)?

Once I have those, I'll check availability and confirm your reservation right away!
================================ Human Message =================================

Around eight in the evening.
================================== Ai Message ==================================

Perfect—8:00 PM for four people on Friday noted! To check availability and finalize your booking, I just need two more details:

- **Which restaurant** would you like to book at?
- **Which Friday** (specific date, or "this Friday")?

Once I have th


Compare against full above to see if anything was cut. Now set the budget far too low, on purpose.

In [36]:
trim_messages(
    full,
    max_tokens = 200,
    token_counter= "approximate",
    strategy="last",
    start_on="human"
)

[HumanMessage(content='Around eight in the evening.', additional_kwargs={}, response_metadata={}, id='cdafba18-c13c-485c-83a5-ebdb40e0ee2f'),
 AIMessage(content='Perfect—8:00 PM for four people on Friday noted! To check availability and finalize your booking, I just need two more details:\n\n- **Which restaurant** would you like to book at?\n- **Which Friday** (specific date, or "this Friday")?\n\nOnce I have those, I\'ll check availability and confirm your reservation right away!', additional_kwargs={'reasoning_content': 'The user wants to book a table for four people on Friday around 8 PM. I still need the restaurant name and specific date. I should ask for those. Let me respond.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants to book a table for four people on Friday around 8 PM. I still need the restaurant name and specific date. I should ask for those. Let me respond.'}]}, response_metadata={'model_name': 'nvidia/nemotron

An empty list, no error, no warning. If one exchange doesn't fit the budget, nothing fits, and you've sent the model a conversation with no conversation in it. A too-small budget looks like this: silent.

Trimming's cost is the beginning of the conversation, the name from turn one is gone. Right when only recent turns matter, wrong the moment something early matters late.

trim_messages also takes include_system=True, which pins a system message so it's never cut. It does nothing here, create_agent applies system_prompt at call time rather than storing it in the list. It matters when you keep your own list, as in section 2

#### 8. Fix two: compress old turns with SummarizationMiddleware
Replace old messages with a short summary instead of dropping them. Middleware does this at a threshold you pick.

In [37]:
from langchain.agents.middleware import SummarizationMiddleware

summarizing = create_agent(
    model = model,
    tools = [],
    system_prompt = "You are a helpful restaurant booking assistant.",
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("messages",6),
            keep= ("messages",2)
        )
    ],checkpointer=InMemorySaver()
)

In [38]:
sum_cfg = {"configurable": {"thread_id": "summary-demo"}}

def chat_sum(text):
    # time.sleep(6)
    result = summarizing.invoke({"messages": [HumanMessage(text)]}, config=sum_cfg)
    return result["messages"][-1].text

chat_sum("My name is Priya and I use a wheelchair.")
chat_sum("I want a table on Friday.")
chat_sum("For four people.")
chat_sum("Around eight in the evening.")
chat_sum("Somewhere quiet, please.")

'Understood—**quiet** is important for a comfortable evening. To find a restaurant that’s both peaceful and fully accessible, I’ll need to know:\n\n- **City or neighborhood** you’re looking in (this is essential for me to search availability and book a table)\n- When you say “quiet,” is there a particular type of area or vibe you have in mind? (e.g., a quiet side street, a cozy neighborhood spot, a quieter district away from the main high street, etc.)\n\nOnce I have the location, I can zero in on accessible restaurants that match your “quiet” preference and have tables open at 8:00 PM for four on Friday.'

In [39]:
summarizing.get_state(sum_cfg).values["messages"]

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nUser Priya (wheelchair user) aims to book an accessible restaurant table for Friday for a party of four, with full accessibility prioritized (step-free entry, wide doorways, accessible restrooms, comfortable seating).\n\n## SUMMARY\n- User name: Priya; uses a wheelchair.\n- Requested: Friday table for four people.\n- AI committed to accessibility-focused search.\n- Still missing: city/neighborhood, specific time on Friday, cuisine/atmosphere preference, additional accessibility details.\n- Conversation progression: user provided name/accessibility need → requested Friday table → specified party size of four.\n\n## ARTIFACTS\nNo files, documents, or external resources were created, modified, or accessed during this conversation. The working context consists of the message exchange above, with key data points: Priya, wheelchair user, Friday, 4 people.\n\n## NEXT STEPS\n- Obtain city/neighborhood f

Look at how many messages are being kept, against how many turns you had.

In [ ]:
# for m in summarising.get_state(sum_cfg).values["messages"]:
#     m.pretty_print()


Summarising costs detail and an extra API call. Exact words are gone, replaced by a paraphrase that might drop what you needed, but early facts survive in some form, which trimming couldn't do.

#### 9. Long-term memory: facts that outlive the conversation
Everything so far is short-term: one conversation, one thread_id. Long-term memory is for facts that outlive the conversation, Priya using a wheelchair is a fact about Priya, still true in March.

The tool is a store: facts keyed by who they belong to.

In [41]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(("profile", "sayali"), "access", {"note":"Sayali uses a wheelchair"})
store.put(("profile", "sayali"), "diet", {"note":"Sayali is vegetarian"})

[item.value for item in store.search(("profile", "sayali"))]


[{'note': 'Sayali uses a wheelchair'}, {'note': 'Sayali is vegetarian'}]

In [42]:
store

("profile", "priya") is a namespace, a labelled drawer. "access" is the key inside it. Two users, two drawers, nothing crosses over.

#### 10. Wiring long-term memory into the assistant
The model only reads messages, so facts go into the system prompt, rebuilt on every call.

In [43]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def with_profile(request):
    facts = request.runtime.store.search(("profile", "sayali"))
    notes = [item.value["note"] for item in facts]
    return "You are a restaurant booking assistant. Known about this guest:\n" + "\n".join(notes)

In [44]:
remembering = create_agent(
    model = model,
    tools = [],
    middleware = [with_profile],
    checkpointer = InMemorySaver(),
    store = store
)

A brand new thread_id: short-term memory is completely empty. No history, nothing.

In [45]:
time.sleep(6)

fresh = {"configurable": {"thread_id": "a-conversation-that-never-happened"}}

result = remembering.invoke(
    {"messages": [HumanMessage("Book me a table for Friday.")]},
    config = fresh
)
result

{'messages': [HumanMessage(content='Book me a table for Friday.', additional_kwargs={}, response_metadata={}, id='c8c00d30-f5da-4ccf-9f06-441691300896'),
  AIMessage(content="I'd be happy to help book a table for Sayali! To make sure the restaurant is fully wheelchair-accessible and has excellent vegetarian options, I'll need a few more details:\n\n- **Which restaurant** would you like to book?\n- **What time** on Friday?\n- **How many people** will be dining?\n- **Which Friday** (date) are you referring to?\n\nOnce I have those, I'll check for accessibility and menu options before confirming the reservation.", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Book me a table for Friday."\n   - I need to identify what\'s missing: restaurant name, time, party size, specific date (just "Friday" - need to know which Friday), and potentially accessibility/dietary requirements based on the known guest info.\n\n2.  **Review 

Two messages in the whole conversation, and it still knew.

Short-term memory: what was said in this conversation.
Long-term memory: what's true about this person, in every conversation.
Mixing them up is how an assistant forgets your name every morning, or brings up last year's complaint at the wrong moment.



#### 11. Save memory to disk with SqliteSaver
InMemorySaver is a Python dictionary, restart the kernel and every conversation is gone. Swap it for a file-backed checkpointer and nothing else about your code changes.

In [48]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect("6_Memory.db", check_same_thread=False)
saver = SqliteSaver(connection)
saver.setup()

durable = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful restaurant booking assistant.",
    checkpointer=saver,
)

In [49]:
# time.sleep(6)

response = durable.invoke(
    {"messages": [HumanMessage("My name is Sayali. Remember it.")]},
    config={"configurable": {"thread_id": "on-disk"}},
)

response

{'messages': [HumanMessage(content='My name is Sayali. Remember it.', additional_kwargs={}, response_metadata={}, id='173ae5f7-f150-4811-a520-28aebb174c7b'),
  AIMessage(content="Nice to meet you, Sayali! I've noted your name. How can I help you with your restaurant booking today?", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "My name is Sayali. Remember it."\n   - This is a simple personal introduction.\n   - The user wants me to remember their name.\n\n2.  **Identify Key Requirements:**\n   - Acknowledge the name "Sayali"\n   - Store/remember it for the conversation context\n   - Respond appropriately as a restaurant booking assistant\n\n3.  **Determine AI Capabilities/Constraints:**\n   - I don\'t have long-term memory across sessions, but within a single conversation, I can remember and reference information.\n   - I should acknowledge the name and use it in future interactions if needed.\n   - I need to stay 

In [50]:
import os

os.path.getsize("6_Memory.db")

4096

That file is on disk now. Restart the kernel, rebuild the agent against the same file, ask for the name on thread on-disk, still there.

Same agent code. Only the checkpointer changed.